# 16 · Profile: O\*NET OnLine "In Demand" software, % of job ads per job (Lightcast)

**Method:** `docs/framework.md` §3. Card: `docs/source-cards/onet_in_demand.md`. Plan: `docs/page3-build-plan.md` §4 item 2 (step 1 of §9).

**Why this source matters.** The job page's "Skills employers ask for" block (design brief §11.10 R1, option B) ranks the software that shows up in a job's ads: "Microsoft Excel 28%, Microsoft Office 17%…" for Accountants. O\*NET 31.0, already staged (`stg_onet__software`), only flags a product In Demand (Y/N) per job, so it can't rank. O\*NET OnLine's per-job "In Demand Software Skills" page adds the number: Percentage = the share of the job's unique U.S. postings that mention the product (Lightcast postings). One CSV per O\*NET-SOC code.

**What was downloaded.** Every O\*NET-SOC code under the 202 named job-page jobs: 254 codes. That's the 204 codes O\*NET 31.0 flags with at least one In Demand product, plus the 50 it doesn't, as the reverse check that those pages really are empty. Two files per code: the CSV, and the HTML page, because only the page states the postings window. 508 requests, 1.5 s apart; `robots.txt` allows `/link/demand/`.

**Questions**
1. What is one row, and do the numbers behave (whole %, sorted, ties)?
2. Does the downloaded product set equal the 31.0 In Demand flags, code by code, both ways (build plan §6: all codes, not 5)?
3. Which postings window, and is it the same on every page?
4. How should several O\*NET profiles under one SOC job become one ranked list?

In [1]:
import re
import sys
import warnings
from pathlib import Path

import pandas as pd
import requests
from IPython.display import Markdown, display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
sys.path.insert(0, str(PROJECT / "src"))
import sources as S
from common import RAW
from explore import q
from job_text import parse_in_demand_page, read_in_demand_csv

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30, "display.max_colwidth", 90, "display.width", 250, "display.max_rows", 80)
md = lambda s: display(Markdown(s))
RETRIEVED = "2026-10-16"   # snapshot date in the file names; a new date re-downloads every page

## 0. The code list and the download

The list comes from the database (the job page's jobs live in the mart), and `fetch()` skips files already on disk, so re-running this cell sends no requests.

In [2]:
codes = q("""select o.onet_soc_code, o.soc_code, o.title as onet_title, jp.soc_title, o.is_specialty
             from staging.stg_onet__occupation o join mart.job_profile jp using (soc_code)
             where not jp.is_all_other order by 1""")
print(f"{len(codes)} O*NET-SOC codes under {codes.soc_code.nunique()} named jobs")
session = requests.Session()
log = pd.DataFrame([S.fetch(src, RETRIEVED, session, pause=1.5) for src in S.onet_in_demand(codes.onet_soc_code.tolist())])
print(log.status.value_counts().to_dict(), "| MB on disk:", round(log.bytes.sum() / 1e6, 1))
man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
print("manifest rows for onet_in_demand:", (man.key == "onet_in_demand").sum(), "| download window (UTC):",
      man.loc[man.key == "onet_in_demand", "downloaded_utc"].agg(["min", "max"]).tolist())

254 O*NET-SOC codes under 198 named jobs
{'already have': 508} | MB on disk: 11.0
manifest rows for onet_in_demand: 508 | download window (UTC): ['2026-10-16T16:45:03+00:00', '2026-10-16T16:58:20+00:00']


**Download findings:** 254 O\*NET-SOC codes sit under 198 of the 202 named jobs. The other 4 are the broad and BLS group codes, which have no O\*NET profile, so their skills block is empty, as it is in `core.fct_job_software`. 508 files (11.0 MB) came down in 13 minutes on 2026-10-16 with no failed request; a re-run finds them all on disk.

## 0b. Source header

| | **O\*NET OnLine, In Demand Software Skills** (`onet_in_demand`) |
|---|---|
| **Publisher** | National Center for O\*NET Development (U.S. DOL, ETA). Percentages from Lightcast job postings |
| **Files** | `skills/onet_in_demand/In_Demand_<code>_2026-10-16.csv` and `.html`, 254 codes × 2 |
| **What a number means** | O\*NET's page: "'Percentage' represents the ratio of unique postings which mention the skill to all unique postings linked to the O\*NET-SOC occupation" |
| **How it was made** | Postings data (Lightcast) coded to O\*NET-SOC; O\*NET lists the products it flags In Demand for the job. Not a survey |
| **License** | O\*NET OnLine content is CC BY 4.0; whether the Lightcast-derived percentages are covered is not confirmed (email to O\*NET pending; gates publishing, not the build) |

## 1. Load: the CSVs and what each page says

Parsers: `src/job_text.py` (`read_in_demand_csv`, `parse_in_demand_page`), shared with the load notebook 36. Each page states how many products it shows and the source line ("Source: Lightcast job postings data for the US nationwide between … and …").

In [3]:
def f(c: str, ext: str) -> Path:
    return RAW / "skills" / "onet_in_demand" / f"In_Demand_{c.replace('.', '-')}_{RETRIEVED}.{ext}"


rows = [read_in_demand_csv(f(c, "csv")).assign(onet_soc_code=c) for c in codes.onet_soc_code]
pages = pd.DataFrame([{"onet_soc_code": c, **parse_in_demand_page(f(c, "html"))} for c in codes.onet_soc_code])
raw = pd.concat(rows, ignore_index=True)
print(f"CSV rows {len(raw):,} over {raw.onet_soc_code.nunique()} codes with any row | pages parsed {len(pages)}")
TEXT = ["Percentage", "Software Skill", "Hot Technology"]
print("CR characters left in any field:", raw[TEXT].apply(lambda s: s.str.contains("\r")).any().any())
display(raw.head(6))
display(pages.head(3))

CSV rows 1,302 over 204 codes with any row | pages parsed 254
CR characters left in any field: False


,Percentage,Software Skill,Hot Technology,csv_order,onet_soc_code
0,12,Microsoft Office software,Yes,1,11-1011.00
1,11,Microsoft Excel,Yes,2,11-1011.00
2,7,Microsoft PowerPoint,Yes,3,11-1011.00
3,5,Microsoft Outlook,Yes,4,11-1011.00
4,12,Microsoft Excel,Yes,1,11-1011.03
5,10,Microsoft Office software,Yes,2,11-1011.03


,onet_soc_code,page_code,page_title,n_shown,vendor,geography,window_start,window_end
0,11-1011.00,11-1011.00,Chief Executives,4,Lightcast,US nationwide,"January 1, 2025","December 31, 2025"
1,11-1011.03,11-1011.03,Chief Sustainability Officers,4,Lightcast,US nationwide,"January 1, 2025","December 31, 2025"
2,11-1021.00,11-1021.00,General and Operations Managers,3,Lightcast,US nationwide,"January 1, 2025","December 31, 2025"


**Load findings:** 1,302 product rows on 204 of the 254 pages; the other 50 CSVs are header-only. Line ends are clean (the CRLF seen in session i's test files doesn't show up in these).

## 2. Grain and page checks

In [4]:
print("(code, product) unique:", not raw.duplicated(["onet_soc_code", "Software Skill"]).any())
print("every page names its own code:", (pages.page_code == pages.onet_soc_code).all(),
      "| page title = O*NET 31.0 title (after dropping the 'Bright Outlook' badge):",
      (pages.merge(codes, on="onet_soc_code").pipe(lambda x: x.page_title == x.onet_title)).mean())
n_csv = raw.groupby("onet_soc_code").size().reindex(pages.onet_soc_code, fill_value=0).values
print("'N software skills shown' on the page = CSV rows:", (pages.n_shown.values == n_csv).all())
print("source line found on every page:", pages.window_start.notna().all(),
      "| vendor:", pages.vendor.unique().tolist(), "| geography:", pages.geography.unique().tolist())
print("windows:", pages.groupby(["window_start", "window_end"]).size().to_dict())

(code, product) unique: True
every page names its own code: True | page title = O*NET 31.0 title (after dropping the 'Bright Outlook' badge): 1.0
'N software skills shown' on the page = CSV rows: True
source line found on every page: True | vendor: ['Lightcast'] | geography: ['US nationwide']
windows: {('January 1, 2025', 'December 31, 2025'): 254}


**Grain and page findings **
- **One row = O\*NET-SOC code × product; the pair is unique.**
- Every page names its own code; its title matches O\*NET 31.0's for all 254, once the "Bright Outlook" badge that sits in the title line is dropped. "N software skills shown" equals the CSV rows on every page.
- Every page states the same source: Lightcast job postings, U.S. nationwide, January 1 to December 31, 2025. So one window applies to the whole table, and the page's footer can state it once.

## 3. Profile: the percentages, the order, ties and names

In [5]:
raw["pct"] = pd.to_numeric(raw.Percentage, errors="raise")
print("whole numbers:", (raw.pct % 1 == 0).all(), "| range:", raw.pct.min(), "to", raw.pct.max(),
      "| Hot Technology values:", raw["Hot Technology"].value_counts().to_dict())
per = raw.groupby("onet_soc_code")
print("products per code (codes with a list):", per.size().describe().round(1).to_dict())
print("CSV sorted by % (high to low) in every code:", per.pct.apply(lambda s: s.is_monotonic_decreasing).all())
cut = per.pct.apply(lambda s: len(s) > 6 and s.iloc[5] == s.iloc[6])
print(f"codes where the 6th and 7th share a % (top-6 cut falls in a tie): {cut.sum()} of {per.ngroups}")
ties = per.pct.apply(lambda s: s.duplicated().any())
print(f"codes with any tie: {ties.sum()}; within a tie O*NET's order is "
      f"alphabetical: {per.apply(lambda g: all(x['Software Skill'].tolist() == sorted(x['Software Skill']) for _, x in g.groupby('pct'))).mean():.2f}")
suffix = raw["Software Skill"].str.extract(r"( software| programming language| \w+ software)$")[0]
print("names ending ' software':", raw["Software Skill"].str.endswith(" software").sum(), "of", len(raw),
      "| distinct products:", raw["Software Skill"].nunique())
display(raw[raw["Software Skill"].str.endswith(" software")]["Software Skill"].value_counts().head(12).to_frame("rows"))

whole numbers: True | range: 4 to 76 | Hot Technology values: {'Yes': 1158, '': 144}
products per code (codes with a list): {'count': 204.0, 'mean': 6.4, 'std': 5.4, 'min': 1.0, '25%': 3.0, '50%': 5.0, '75%': 8.0, 'max': 38.0}
CSV sorted by % (high to low) in every code: True
codes where the 6th and 7th share a % (top-6 cut falls in a tie): 20 of 204
codes with any tie: 105; within a tie O*NET's order is alphabetical: 0.62
names ending ' software': 306 of 1302 | distinct products: 206


,rows
Software Skill,
Microsoft Office software,178
Adobe Creative Cloud software,18
SAP software,16
Salesforce software,14
Microsoft Azure software,14
Amazon Web Services AWS software,13
ESRI ArcGIS software,13
Workday software,5
Firewall software,5


**Profile findings**
- Percentages are whole numbers from 4 to 76, and every list comes sorted high to low. A code has 1 to 38 products (median 5).
- **Ties are common:** 105 codes have at least one, and in 20 of them the 6th and 7th products share a %, so the top-6 cut falls inside a tie. Within a tie O\*NET's order is alphabetical only 62% of the time, so it most likely follows the unrounded share. The rank keeps O\*NET's order as the tie-break (build plan §4 item 5), which makes the cut deterministic.
- `Hot Technology` is "Yes" or blank (144 rows), never "No"; blank = not a Hot Technology.
- 306 of 1,302 names end in " software" ("Microsoft Office software" alone 178 times). Dropping it gives the mockup's names ("Microsoft Office", "SAP").

## 4. Tie-out: the product set equals the O\*NET 31.0 In Demand flags, code by code

`stg_onet__software` can list a product twice under one code (two UNSPSC commodities), so the flags are OR'd per (code, product) first, as `core.fct_job_software` does.

In [6]:
sw = q("""select onet_soc_code, product, bool_or(is_in_demand) as in_demand, bool_or(is_hot_technology) as hot
          from staging.stg_onet__software where onet_soc_code = any(%(c)s) group by 1, 2""", c=codes.onet_soc_code.tolist())
flag = sw[sw.in_demand].groupby("onet_soc_code").product.apply(set)
live = raw.groupby("onet_soc_code")["Software Skill"].apply(set)
cmp_ = pd.DataFrame({"flags_31": flag, "live": live}).reindex(codes.onet_soc_code)
cmp_ = cmp_.map(lambda v: v if isinstance(v, set) else set())
cmp_["equal"] = cmp_.flags_31 == cmp_.live
print(f"codes compared: {len(cmp_)} | equal sets: {cmp_.equal.sum()} | flagged in 31.0: {(cmp_.flags_31.map(len) > 0).sum()}"
      f" | with a live list: {(cmp_.live.map(len) > 0).sum()}")
bad = cmp_[~cmp_.equal]
for c, r in bad.iterrows():
    print(f"  {c}: only live {sorted(r.live - r.flags_31)} | only 31.0 {sorted(r.flags_31 - r.live)}")
hot = raw.merge(sw, left_on=["onet_soc_code", "Software Skill"], right_on=["onet_soc_code", "product"], how="left")
print("live 'Hot Technology' = 31.0 hot flag:", ((hot["Hot Technology"] == "Yes") == hot.hot).mean())

codes compared: 254 | equal sets: 254 | flagged in 31.0: 204 | with a live list: 204


live 'Hot Technology' = 31.0 hot flag: 1.0


Tie-out (build plan §6, on all codes rather than 5): **254 of 254 pages list exactly the products O\*NET 31.0 flags In Demand for that code.** The 204 flagged codes all have a list, and the 50 unflagged pages show none. The Hot Technology mark agrees with 31.0 on all 1,302 rows. So the live site and the staged 31.0 release are the same edition of the flags, and the percentages add only the ranking.

## 5. From O\*NET profiles to SOC jobs

The job page shows SOC jobs. 31.0's flags were OR'd across a SOC job's profiles (`core.fct_job_software`), but percentages from different posting pools can't be added or averaged into one share of ads. So which profile's list does a SOC job show?

In [7]:
lists = raw.groupby("onet_soc_code").size().rename("n_products")
cj = codes.merge(lists, left_on="onet_soc_code", right_index=True, how="left").fillna({"n_products": 0})
cj["dot00"] = cj.onet_soc_code.str.endswith(".00")
soc = cj.groupby("soc_code").agg(profiles=("onet_soc_code", "size"), has_00=("dot00", "any"),
                                 lists=("n_products", lambda s: (s > 0).sum()),
                                 list_00=("n_products", lambda s: (s[cj.loc[s.index, "dot00"]] > 0).any()))
print("named jobs:", len(soc), "| with any list:", (soc.lists > 0).sum(), "| with a .00 list:", soc.list_00.sum())
display(soc.groupby(["profiles", "has_00", "lists", "list_00"]).size().to_frame("jobs"))
md("**Jobs with a list but none on a `.00` profile:**")
display(cj[cj.soc_code.isin(soc.index[(soc.lists > 0) & ~soc.list_00])][["soc_code", "soc_title", "onet_soc_code", "onet_title", "n_products"]])

named jobs: 198 | with any list: 164 | with a .00 list: 162


jobs
profiles has_00 lists list_00      
1        True   0     False      33
                1     True      134
2        True   0     False       1
                1     False       1
                      True        5
                2     True       11
3        True   2     True        1
                3     True        6
4        True   4     True        3
5        True   1     False       1
6        True   2     True        1
7        True   7     True        1

**Jobs with a list but none on a `.00` profile:**

,soc_code,soc_title,onet_soc_code,onet_title,n_products
128,17-3024,Electro-Mechanical and Mechatronics Technologists and Technicians,17-3024.00,Electro-Mechanical and Mechatronics Technologists and Technicians,0.0
129,17-3024,Electro-Mechanical and Mechatronics Technologists and Technicians,17-3024.01,Robotics Technicians,5.0
221,29-1141,Registered Nurses,29-1141.00,Registered Nurses,0.0
222,29-1141,Registered Nurses,29-1141.01,Acute Care Nurses,0.0
223,29-1141,Registered Nurses,29-1141.02,Advanced Practice Psychiatric Nurses,2.0
224,29-1141,Registered Nurses,29-1141.03,Critical Care Nurses,0.0
225,29-1141,Registered Nurses,29-1141.04,Clinical Nurse Specialists,0.0


Where a job has a `.00` list and specialty lists too, how different are they? (Top 6 of each, by %.)

In [8]:
multi = soc.index[soc.list_00 & (soc.lists > 1)]
top = raw.sort_values(["onet_soc_code", "csv_order"]).groupby("onet_soc_code").head(6)
out = []
for s_ in multi:
    cs = cj[(cj.soc_code == s_) & (cj.n_products > 0)].onet_soc_code
    base = set(top[top.onet_soc_code == f"{s_}.00"]["Software Skill"])
    for c in cs[cs != f"{s_}.00"]:
        other = set(top[top.onet_soc_code == c]["Software Skill"])
        out.append({"soc_code": s_, "specialty": c, "title": cj.set_index("onet_soc_code").onet_title[c],
                    "top6_shared_with_00": len(base & other), "of": len(other)})
display(pd.DataFrame(out))

,soc_code,specialty,title,top6_shared_with_00,of
0,11-1011,11-1011.03,Chief Sustainability Officers,4,4
1,11-3013,11-3013.01,Security Managers,4,5
2,11-3031,11-3031.01,Treasurers and Controllers,4,5
3,11-3031,11-3031.03,Investment Fund Managers,3,3
4,11-3051,11-3051.01,Quality Control Systems Managers,4,4
5,11-9121,11-9121.01,Clinical Research Coordinators,3,4
6,11-9121,11-9121.02,Water Resource Specialists,2,4
7,13-1041,13-1041.01,Environmental Compliance Inspectors,3,3
8,13-1041,13-1041.03,Equal Opportunity Representatives and Officers,2,2
9,13-1041,13-1041.04,Government Property Inspectors and Investigators,3,5


**Roll-up findings**
- 164 named jobs have a list on at least one profile (the plan's count, which ORs the flags across profiles), and 162 have one on their `.00` profile.
- The two without a `.00` list: Registered Nurses (only *Advanced Practice Psychiatric Nurses* has one: Microsoft Teams 7%, Google Meet 7%) and 17-3024 Electro-Mechanical and Mechatronics Technologists (only *Robotics Technicians*).
- Where a job has a `.00` list and specialty lists, they differ: specialties share 0 to 6 of their top 6 with the `.00` list (*Video Game Designers* shares none with *Web and Digital Interface Designers*). Percentages from different posting pools can't be added or averaged into one share of ads.

## 6. What the page would show: top 6 per job, and the mockup tie-out

**Proposed rule** (built at step 2 in `job_skill_demand`): a job shows its `.00` profile's list only. A specialty page counts the postings of a narrower title, so its percentages aren't the job's (§5: specialties share as few as 0 of 6 top products with the `.00` list), and the two jobs whose only list is a specialty's would mislead (Registered Nurses through *Advanced Practice Psychiatric Nurses*: Microsoft Teams 7%, Google Meet 7%). Ranked by % (high to low), then O\*NET's order; up to 6 rows; display name = O\*NET's name minus a trailing " software".

In [9]:
chosen = cj[cj.dot00 & (cj.n_products > 0)]
show = (raw.merge(chosen[["onet_soc_code", "soc_code"]], on="onet_soc_code")
           .sort_values(["soc_code", "pct", "csv_order"], ascending=[True, False, True]))
show["rank"] = show.groupby("soc_code").cumcount() + 1
show = show[show["rank"] <= 6].assign(product_display=lambda x: x["Software Skill"].str.replace(r" software$", "", regex=True))
print("named jobs with a list on the page:", show.soc_code.nunique(), "| rows:", len(show),
      "| jobs with a specialty list only (empty state under the rule):", sorted(set(soc.index[soc.lists > 0]) - set(show.soc_code)))
fmt = lambda s_: ", ".join(f"{r.product_display} {r.pct:.0f}%" for r in show[show.soc_code == s_].itertuples())
for s_ in ["13-2011", "15-2051", "15-1211", "13-1111", "25-2021"]:
    print(f"{s_}: {fmt(s_) or '(none: empty state)'}")
assert fmt("13-2011") == ("Microsoft Excel 28%, Microsoft Office 17%, Intuit QuickBooks 8%, SAP 6%, "
                          "Microsoft PowerPoint 6%, Microsoft Outlook 6%")   # mockup v4.1 and build plan §5
long = show[show.product_display.str.len() > 32]
print(f"display names over 32 characters in a top 6: {long.product_display.nunique()} names in {long.soc_code.nunique()} jobs; longest "
      f"{show.product_display.str.len().max()}: {show.loc[show.product_display.str.len().idxmax(), 'product_display']}")
display(long.drop_duplicates("product_display")[["soc_code", "product_display"]].assign(chars=lambda x: x.product_display.str.len()))

named jobs with a list on the page: 162 | rows: 731 | jobs with a specialty list only (empty state under the rule): ['17-3024', '29-1141']
13-2011: Microsoft Excel 28%, Microsoft Office 17%, Intuit QuickBooks 8%, SAP 6%, Microsoft PowerPoint 6%, Microsoft Outlook 6%
15-2051: Python 66%, Structured query language SQL 51%, R 34%, Tableau 22%, Microsoft Power BI 19%, Amazon Web Services AWS 17%
15-1211: Structured query language SQL 16%, Microsoft Excel 15%, Microsoft Office 14%, SAP 13%, Microsoft PowerPoint 10%, ServiceNow 6%
13-1111: Microsoft Excel 21%, Structured query language SQL 20%, Microsoft PowerPoint 14%, Microsoft Office 14%, Microsoft Power BI 11%, Atlassian JIRA 10%
25-2021: (none: empty state)
display names over 32 characters in a top 6: 8 names in 13 jobs; longest 86: Very high speed integrated circuit VHSIC hardware description language VHDL simulation


,soc_code,product_display,chars
44,11-3021,Enterprise application integration EAI,38
242,13-2081,Tax compliance property tax management,38
489,15-2011,Microsoft Visual Basic for Applications VBA,43
551,17-1021,Geographic information system GIS systems,41
590,17-2061,Very high speed integrated circuit VHSIC hardware description language VHDL simulation,86
783,19-3092,Geographic information system GIS,33
815,19-4092,Guidance Software EnCase Enterprise,35
843,23-2011,Orion Law Management Systems Orion,34


**Result **
- 162 named jobs get a ranked list (731 rows, up to 6 each); 40 get the empty state. That's the plan's 38, plus Registered Nurses and 17-3024 under the `.00` rule. This changes one number in the build plan (§2: 164 → 162), flagged in the card.
- 13-2011 matches the mockup v4.1 and build plan §5 exactly: Microsoft Excel 28%, Microsoft Office 17%, Intuit QuickBooks 8%, SAP 6%, Microsoft PowerPoint 6%, Microsoft Outlook 6%. 25-2021 is the empty state, as mocked.
- R1's test case reads as hoped: Data Scientists Python 66%, SQL 51%, R 34%, Tableau 22%, Power BI 19%. Management Analysts lead with Excel 21% and SQL 20%.
- 8 display names run past 32 characters (13 jobs; the longest is 86, a VHDL tool for Electronics Engineers). That's the same truncation question as the long major names (build plan §6, Desktop test 6).
- O\*NET writes some names with the acronym after the words ("Structured query language SQL", "Amazon Web Services AWS"). They're kept as O\*NET stores them, the same as the mockup's names.